# Córdoba — Geographic Generalization and Few-Shot Adaptation

**Environment**: Colab A100 (GPU required)

**Part 1**: Zero-shot generalization — base model (trained on Corrientes) evaluated on Córdoba
**Part 2**: Few-shot domain adaptation — decoder fine-tuned on 100 Córdoba patches

In [ ]:
import os, random, warnings
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU   : {torch.cuda.get_device_name(0)}')

# â”€â”€ Paths â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
if IN_COLAB:
    BASE = Path('/content/drive/MyDrive/GeoAI/wildfire-spread')
else:
    BASE = Path('G:/Mon Drive/GeoAI/wildfire-spread')

CKPT_DIR   = BASE / 'models'
RESULTS    = BASE / 'results'
RESULTS.mkdir(exist_ok=True)

# Patches de CÃ³rdoba â€” copiados desde local a Drive en el paso anterior
if IN_COLAB:
    LOCAL_CORDOBA = Path('/content/cordoba_patches')
    if not (LOCAL_CORDOBA / 'images').exists():
        print('Copiando patches CÃ³rdoba a /content/...')
        import subprocess
        LOCAL_CORDOBA.mkdir(exist_ok=True)
        subprocess.run(['cp', '-r', str(BASE / 'data/cordoba/patches/images'),     str(LOCAL_CORDOBA)], check=True)
        subprocess.run(['cp', '-r', str(BASE / 'data/cordoba/patches/masks_dnbr'), str(LOCAL_CORDOBA)], check=True)
        print('Copy complete.')
    IMG_DIR  = LOCAL_CORDOBA / 'images'
    MASK_DIR = LOCAL_CORDOBA / 'masks_dnbr'
else:
    IMG_DIR  = BASE / 'data/cordoba/patches/images'
    MASK_DIR = BASE / 'data/cordoba/patches/masks_dnbr'

img_paths  = sorted(IMG_DIR.glob('*.npy'))
mask_paths = sorted(MASK_DIR.glob('*.npy'))
assert len(img_paths) == len(mask_paths), f'{len(img_paths)} imgs vs {len(mask_paths)} masks'
print(f'\nPatches CÃ³rdoba: {len(img_paths)}')
fire_flags = np.array([np.load(f, mmap_mode='r').max() > 0 for f in mask_paths])
print(f'Patches con cicatriz: {fire_flags.sum()} ({fire_flags.mean()*100:.1f}%)')

In [ ]:
# ── Architecture — identical to notebook 04 (v1.5) ───────────────────────────────
from terratorch.registry import BACKBONE_REGISTRY

PRITHVI_MEAN = np.array([0.033349, 0.057012, 0.058897, 0.232325, 0.197285, 0.119449], dtype=np.float32)
PRITHVI_STD  = np.array([0.022691, 0.026808, 0.040041, 0.077917, 0.087087, 0.072420], dtype=np.float32)
BAND_IDX     = [0, 1, 2, 4, 5, 6]   # B02, B03, B04, B8A, B11, B12
IMG_SIZE     = 224
T            = (256 - IMG_SIZE) // 2
THRESHOLD    = 0.525

class MultiScaleNeck(nn.Module):
    def __init__(self, n_side=14, d_model=768, fpn_ch=256):
        super().__init__()
        self.n_side = n_side
        self.lateral = nn.ModuleList([
            nn.Sequential(nn.Conv2d(d_model, fpn_ch, 1, bias=False),
                          nn.BatchNorm2d(fpn_ch), nn.GELU()) for _ in range(4)
        ])
        self.td_conv = nn.ModuleList([
            nn.Sequential(nn.Conv2d(fpn_ch, fpn_ch, 3, padding=1, bias=False),
                          nn.BatchNorm2d(fpn_ch), nn.GELU()) for _ in range(3)
        ])
    def tok2map(self, tokens):
        B, L, D = tokens.shape
        return tokens.permute(0, 2, 1).reshape(B, D, self.n_side, self.n_side)
    def forward(self, layers_out):
        feats = [self.lateral[i](self.tok2map(layers_out[idx][:, 1:, :]))
                 for i, idx in enumerate([2, 5, 8, 11])]
        out = feats[3]
        out = self.td_conv[0](feats[2] + out)
        out = self.td_conv[1](feats[1] + out)
        out = self.td_conv[2](feats[0] + out)
        return out  # (B, 256, 14, 14)


class FPNDecoder(nn.Module):
    def __init__(self, in_ch=256):
        super().__init__()
        self.up = nn.Sequential(
            nn.ConvTranspose2d(in_ch, 256, 2, stride=2), nn.BatchNorm2d(256), nn.GELU(),
            nn.ConvTranspose2d(256,   128, 2, stride=2), nn.BatchNorm2d(128), nn.GELU(),
            nn.ConvTranspose2d(128,    64, 2, stride=2), nn.BatchNorm2d(64),  nn.GELU(),
            nn.ConvTranspose2d(64,     32, 2, stride=2), nn.BatchNorm2d(32),  nn.GELU(),
            nn.Conv2d(32, 1, 1),
        )
    def forward(self, x): return self.up(x)


class PrithviSegModelV2(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = BACKBONE_REGISTRY.build('prithvi_eo_v1_100')
        self.neck     = MultiScaleNeck(n_side=14, d_model=768, fpn_ch=256)
        self.decoder  = FPNDecoder(in_ch=256)
    def forward(self, x):
        return self.decoder(self.neck(self.backbone(x.unsqueeze(2))))

print('Loading model v1.5...')
model = PrithviSegModelV2().to(DEVICE)
ckpt_path = CKPT_DIR / 'best_prithvi_burn_scar_wildfire.pth'
if not ckpt_path.exists():
    candidates = list(CKPT_DIR.glob('*prithvi*.pth'))
    if candidates:
        ckpt_path = candidates[0]
    else:
        raise FileNotFoundError(f'No checkpoint found in {CKPT_DIR}')
model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
model.eval()
print(f'Model loaded: {ckpt_path.name}')

In [ ]:
# â”€â”€ Dataset de CÃ³rdoba â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
class CordobaDataset(Dataset):
    def __init__(self, img_paths, mask_paths):
        self.img_paths  = img_paths
        self.mask_paths = mask_paths

    def __len__(self): return len(self.img_paths)

    def __getitem__(self, idx):
        img  = np.load(self.img_paths[idx]).astype(np.float32)    # (11, 256, 256)
        mask = np.load(self.mask_paths[idx]).astype(np.float32)   # (256, 256)
        img  = img[BAND_IDX]               # (6, 256, 256)
        img /= 10000.0
        img  = (img - PRITHVI_MEAN[:, None, None]) / PRITHVI_STD[:, None, None]
        t    = (256 - IMG_SIZE) // 2
        img  = img[:, t:t+IMG_SIZE, t:t+IMG_SIZE]
        mask = mask[t:t+IMG_SIZE, t:t+IMG_SIZE]
        return torch.from_numpy(img), torch.from_numpy(mask).unsqueeze(0)

ds     = CordobaDataset(img_paths, mask_paths)
loader = DataLoader(ds, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)
print(f'Batches a evaluar: {len(loader)}')

In [ ]:
# â”€â”€ Inferencia completa â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
all_probs, all_targets = [], []
patch_probs_list, patch_masks_list, patch_imgs_list = [], [], []

print('Evaluando en CÃ³rdoba...')
with torch.no_grad():
    for imgs, masks in tqdm(loader):
        imgs  = imgs.to(DEVICE)
        probs = model(imgs).sigmoid().squeeze(1).cpu().numpy()   # (B, 224, 224)
        masks_np = masks.squeeze(1).cpu().numpy()                # (B, 224, 224)

        all_probs.append(probs.reshape(-1))
        all_targets.append(masks_np.reshape(-1))

        for b in range(len(imgs)):
            patch_probs_list.append(probs[b])
            patch_masks_list.append(masks_np[b])
            patch_imgs_list.append(imgs[b].cpu().numpy())

all_probs   = np.concatenate(all_probs)
all_targets = np.concatenate(all_targets).astype(np.int32)
all_preds   = (all_probs > THRESHOLD).astype(np.int32)

# â”€â”€ MÃ©tricas â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
precision, recall, f1, _ = precision_recall_fscore_support(
    all_targets, all_preds, pos_label=1, average='binary', zero_division=0)
tp = int(((all_preds==1)&(all_targets==1)).sum())
fp = int(((all_preds==1)&(all_targets==0)).sum())
fn = int(((all_preds==0)&(all_targets==1)).sum())
tn = int(((all_preds==0)&(all_targets==0)).sum())
iou_fire = tp / (tp + fp + fn + 1e-6)

try:
    auc = roc_auc_score(all_targets, all_probs)
except:
    auc = float('nan')

print('\n' + '='*55)
print('  RESULTADOS â€” CÃ“RDOBA (Test geogrÃ¡fico)')
print('='*55)
print(f'  Precision (cicatriz): {precision:.4f}')
print(f'  Recall (cicatriz)   : {recall:.4f}')
print(f'  F1-Score            : {f1:.4f}')
print(f'  IoU (cicatriz)      : {iou_fire:.4f}')
print(f'  AUC-ROC             : {auc:.4f}')
print('â”€'*55)
print('  COMPARATIVA:')
print(f'  Corrientes val IoU  : 0.4198  â† entrenamiento')
print(f'  CÃ³rdoba test IoU    : {iou_fire:.4f}  â† este notebook')
print('='*55)

In [ ]:
# â”€â”€ Curva Precision-Recall â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
from sklearn.metrics import precision_recall_curve, roc_curve

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# PR curve
prec_c, rec_c, thr_c = precision_recall_curve(all_targets, all_probs)
axes[0].plot(rec_c, prec_c, color='crimson', lw=2)
axes[0].scatter([recall], [precision], color='black', zorder=5, s=80,
                label=f'thr=0.5  |  F1={f1:.3f}')
axes[0].set_xlabel('Recall'); axes[0].set_ylabel('Precision')
axes[0].set_title('Precision-Recall â€” CÃ³rdoba test')
axes[0].legend(); axes[0].grid(alpha=0.3)

# ROC curve
if not np.isnan(auc):
    fpr_c, tpr_c, _ = roc_curve(all_targets, all_probs)
    axes[1].plot(fpr_c, tpr_c, color='steelblue', lw=2, label=f'AUC={auc:.3f}')
    axes[1].plot([0,1],[0,1],'--',color='gray')
    axes[1].set_xlabel('False Positive Rate'); axes[1].set_ylabel('True Positive Rate')
    axes[1].set_title('ROC â€” CÃ³rdoba test')
    axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('EvaluaciÃ³n de generalizaciÃ³n geogrÃ¡fica â€” CÃ³rdoba Oct 2020', fontsize=12)
plt.tight_layout()
out = RESULTS / 'cordoba_evaluation_curves.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

In [ ]:
# â”€â”€ VisualizaciÃ³n de predicciones â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def norm_band(x, p=2):
    v = x[x > 0]
    if not len(v): return np.zeros_like(x)
    lo, hi = np.percentile(v, [p, 100-p])
    return np.clip((x - lo) / (hi - lo + 1e-6), 0, 1)

# IoU por patch
patch_ious = []
for prob, mask in zip(patch_probs_list, patch_masks_list):
    pred = (prob > THRESHOLD).astype(np.float32)
    tp_ = (pred*mask).sum(); fp_ = (pred*(1-mask)).sum(); fn_ = ((1-pred)*mask).sum()
    patch_ious.append(float(tp_/(tp_+fp_+fn_+1e-6)))
patch_ious = np.array(patch_ious)

fire_patch_idx = [i for i,f in enumerate(fire_flags) if f]
best_idx       = sorted(fire_patch_idx, key=lambda i: patch_ious[i], reverse=True)[:4]

fig, axes = plt.subplots(4, 4, figsize=(16, 16))
col_titles = ['RGB', 'Ground truth (dNBR)', 'PredicciÃ³n (prob)', 'TP/FP/FN']
for ci, ct in enumerate(col_titles):
    axes[0, ci].set_title(ct, fontsize=10, pad=6)

for row, vi in enumerate(best_idx):
    img_np   = patch_imgs_list[vi]     # (6, 224, 224)
    prob_np  = patch_probs_list[vi]    # (224, 224)
    mask_np  = patch_masks_list[vi]    # (224, 224)
    pred_bin = (prob_np > THRESHOLD).astype(np.float32)
    iou_v    = patch_ious[vi]

    def denorm(ch):
        return norm_band(img_np[ch] * PRITHVI_STD[ch] + PRITHVI_MEAN[ch])
    rgb = np.dstack([denorm(2), denorm(1), denorm(0)])  # B04, B03, B02

    err = np.zeros((*mask_np.shape, 3))
    err[(pred_bin==1)&(mask_np==1)] = [0.0, 0.8, 0.0]
    err[(pred_bin==1)&(mask_np==0)] = [1.0, 0.5, 0.0]
    err[(pred_bin==0)&(mask_np==1)] = [0.9, 0.1, 0.1]

    axes[row,0].imshow(rgb);                               axes[row,0].axis('off')
    axes[row,1].imshow(mask_np,  cmap='Reds', vmin=0, vmax=1); axes[row,1].axis('off')
    axes[row,2].imshow(prob_np,  cmap='hot',  vmin=0, vmax=1); axes[row,2].axis('off')
    axes[row,3].imshow(err);                               axes[row,3].axis('off')
    axes[row,0].set_ylabel(f'IoU={iou_v:.3f}', fontsize=9)

tp_p  = mpatches.Patch(color=[0,.8,0], label='TP (detectado)')
fp_p  = mpatches.Patch(color=[1,.5,0], label='FP (falsa alarma)')
fn_p  = mpatches.Patch(color=[.9,.1,.1], label='FN (perdido)')
fig.legend(handles=[tp_p,fp_p,fn_p], loc='lower center', ncol=3, fontsize=10,
           bbox_to_anchor=(0.5, -0.01))

plt.suptitle(
    f'CÃ³rdoba Test  |  IoU={iou_fire:.4f}  |  Recall={recall:.4f}  |  Precision={precision:.4f}\n'
    f'Modelo entrenado en Corrientes (2022) â€” evaluado en CÃ³rdoba (2020)',
    fontsize=11, y=1.01)
plt.tight_layout()
out = RESULTS / 'cordoba_predictions.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out}')

In [ ]:
# â”€â”€ Resumen final para portfolio â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print('\n' + '='*60)
print('  PORTFOLIO SUMMARY â€” Wildfire Burn Scar Detection')
print('='*60)
print('  Modelo: Prithvi-100M (IBM/NASA) + FPN decoder')
print('  Encoder: frozen (100M params pre-entrenados en HLS/S2)')
print('  Decoder: entrenado en Corrientes, Argentina (dic 2021â€“feb 2022)')
print('â”€'*60)
print('  TRAINING SET (Corrientes):')
print('    Ãrea   : -59.5W/-29.0S â†’ -56.0W/-26.5N (pastizales/humedales)')
print('    Labels : dNBR > 0.10 (cicatrices de incendio)')
print('    Patches: 5,687 Ã— 256Ã—256 px Ã— 6 bandas S2')
print('â”€'*60)
print('  TEST SET (CÃ³rdoba) â€” zona NUNCA vista:')
print(f'    Ãrea   : -65.5W/-33.0S â†’ -62.5W/-30.5N (sierras/monte xerÃ³filo)')
print(f'    Patches: {len(img_paths)}')
print(f'    IoU cicatriz : {iou_fire:.4f}')
print(f'    Recall       : {recall:.4f}')
print(f'    Precision    : {precision:.4f}')
print('â”€'*60)
print('  PIPELINE COMPLETO:')
print('    S2 L2A (CDSE) â†’ dNBR labels â†’ Prithvi fine-tune â†’ GeneralizaciÃ³n')
print('    Corrientes 2022 â†’ CÃ³rdoba 2020 = diferente regiÃ³n + diferente aÃ±o')
print('='*60)

---

## Part 2 — Few-Shot Decoder Fine-Tuning (100 Córdoba Patches)

In [ ]:
# ── Split: 100 fine-tune patches / resto test ─────────────────────────────────
print('Scanning fire flags...')
fire_flags = np.array([
    np.load(p, mmap_mode='r').max() > 0 for p in tqdm(mask_paths)
], dtype=np.int32)

indices = np.arange(len(img_paths))

# Separar 100 patches para fine-tuning (estratificados)
ft_idx, test_idx = train_test_split(
    indices,
    train_size=N_FINETUNE,
    stratify=fire_flags,
    random_state=SEED
)

print(f'Fine-tune : {len(ft_idx):>5} patches ({fire_flags[ft_idx].sum()} positive, '
      f'{fire_flags[ft_idx].mean()*100:.1f}%)')
print(f'Test      : {len(test_idx):>5} patches ({fire_flags[test_idx].sum()} positive, '
      f'{fire_flags[test_idx].mean()*100:.1f}%)')

ft_ds   = WildfireDataset([img_paths[i] for i in ft_idx],   [mask_paths[i] for i in ft_idx],   augment=True)
test_ds = WildfireDataset([img_paths[i] for i in test_idx], [mask_paths[i] for i in test_idx], augment=False)

ft_loader   = DataLoader(ft_ds,   batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f'Fine-tune batches: {len(ft_loader)}  |  Test batches: {len(test_loader)}')

In [ ]:
# ── Evaluación ANTES del fine-tuning (baseline Córdoba) ──────────────────────
def evaluate(model, loader, threshold=THRESHOLD):
    model.eval()
    all_probs, all_targets = [], []
    with torch.no_grad():
        for imgs, masks in tqdm(loader, desc='Evaluating', leave=False):
            probs    = model(imgs.to(DEVICE)).sigmoid().squeeze(1).cpu().numpy()
            masks_np = masks.squeeze(1).cpu().numpy()
            all_probs.append(probs.reshape(-1))
            all_targets.append(masks_np.reshape(-1))
    all_probs   = np.concatenate(all_probs)
    all_targets = np.concatenate(all_targets).astype(np.int32)
    all_preds   = (all_probs > threshold).astype(np.int32)
    prec, rec, f1, _ = precision_recall_fscore_support(
        all_targets, all_preds, pos_label=1, average='binary', zero_division=0)
    tp = int(((all_preds==1)&(all_targets==1)).sum())
    fp = int(((all_preds==1)&(all_targets==0)).sum())
    fn = int(((all_preds==0)&(all_targets==1)).sum())
    iou = tp / (tp + fp + fn + 1e-6)
    try:
        auc = roc_auc_score(all_targets, all_probs)
    except Exception:
        auc = float('nan')
    return dict(iou=iou, precision=prec, recall=rec, f1=f1, auc=auc)


print('Evaluating BASE model (no fine-tuning) on Córdoba test set...')
results_before = evaluate(model, test_loader)
print(f'  IoU       : {results_before["iou"]:.4f}')
print(f'  Recall    : {results_before["recall"]:.4f}')
print(f'  Precision : {results_before["precision"]:.4f}')
print(f'  AUC-ROC   : {results_before["auc"]:.4f}')

In [ ]:
# ── Decoder fine-tuning on 100 Córdoba patches ─────────────────────
dice_loss  = DiceLoss(mode='binary', from_logits=True)
focal_loss = FocalLoss(mode='binary', gamma=2.0, alpha=0.75)

def criterion(pred, target):
    return dice_loss(pred, target) + focal_loss(pred, target)

# Solo entrenar neck + decoder (encoder sigue congelado)
trainable = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable, lr=LR_FT, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS_FT, eta_min=1e-7)

best_iou_ft  = 0.0
ft_losses, ft_ious = [], []

print(f'Fine-tuning decoder — {EPOCHS_FT} epochs | LR={LR_FT} | {N_FINETUNE} Córdoba patches')
for epoch in range(1, EPOCHS_FT + 1):
    model.train()
    model.backbone.eval()   # encoder always frozen
    ep_loss = 0.0
    for imgs, masks in tqdm(ft_loader, desc=f'FT {epoch:02d}/{EPOCHS_FT}', leave=False):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(imgs), masks)
        loss.backward(); optimizer.step()
        ep_loss += loss.item()
    scheduler.step()
    ft_losses.append(ep_loss / len(ft_loader))

    # Validar sobre test set
    res = evaluate(model, test_loader)
    ft_ious.append(res['iou'])

    print(f'Epoch {epoch:02d} | Loss: {ft_losses[-1]:.4f} | IoU: {ft_ious[-1]:.4f}')

    if ft_ious[-1] > best_iou_ft:
        best_iou_ft = ft_ious[-1]
        torch.save(model.state_dict(), CKPT_FT)
        print(f'  → Saved (IoU: {best_iou_ft:.4f})')

print(f'\nBest IoU fine-tuned: {best_iou_ft:.4f}')

In [ ]:
# ── Evaluación DESPUÉS del fine-tuning ───────────────────────────────────────
model.load_state_dict(torch.load(CKPT_FT, map_location=DEVICE))
results_after = evaluate(model, test_loader)

print('\n' + '='*60)
print('  COMPARISON — Córdoba test set')
print('='*60)
print(f'  {"Métrica":<12} {"Base (07)":>12} {"Fine-tuned":>12} {"Mejora":>10}')
print('-'*60)
for key in ['iou', 'recall', 'precision', 'auc']:
    label = {'iou': 'IoU', 'recall': 'Recall', 'precision': 'Precision', 'auc': 'AUC-ROC'}[key]
    before = results_before[key]
    after  = results_after[key]
    delta  = after - before
    print(f'  {label:<12} {before:>12.4f} {after:>12.4f} {delta:>+10.4f}')
print('='*60)
print(f'\nFine-tuned model saved: {CKPT_FT.name}')

In [ ]:
# ── Curvas de fine-tuning ─────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(range(1, EPOCHS_FT+1), ft_losses, color='steelblue', label='Fine-tune loss')
axes[0].set(xlabel='Epoch', ylabel='Loss', title='Loss — fine-tuning Córdoba')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(range(1, EPOCHS_FT+1), ft_ious, color='green', label='IoU Córdoba')
axes[1].axhline(results_before['iou'], color='gray', ls=':', label=f'Base: {results_before["iou"]:.4f}')
axes[1].axhline(best_iou_ft,           color='red',  ls='--', label=f'Best FT: {best_iou_ft:.4f}')
axes[1].set(xlabel='Epoch', ylabel='IoU', title='IoU Córdoba — fine-tuning')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
out = RESULTS / 'cordoba_finetune_curves.png'
plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
print(f'Saved: {out}')

In [ ]:
# ── Visualización — predicciones del modelo fine-tuned ───────────────────────
def norm_band(x, p=2):
    v = x[x > 0]
    if not len(v): return np.zeros_like(x)
    lo, hi = np.percentile(v, [p, 100-p])
    return np.clip((x - lo) / (hi - lo + 1e-6), 0, 1)

model.eval()
sample_imgs, sample_masks, sample_probs = [], [], []
with torch.no_grad():
    for imgs, masks in test_loader:
        probs = model(imgs.to(DEVICE)).sigmoid().squeeze(1).cpu().numpy()
        for b in range(len(imgs)):
            m = masks[b].squeeze().numpy()
            if m.max() > 0:
                sample_imgs.append(imgs[b].numpy())
                sample_masks.append(m)
                sample_probs.append(probs[b])
        if len(sample_imgs) >= 3:
            break

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for row in range(min(3, len(sample_imgs))):
    img_np  = sample_imgs[row]
    prob_np = sample_probs[row]
    mask_np = sample_masks[row]
    pred_b  = (prob_np > THRESHOLD).astype(np.float32)

    def dn(ch):
        return norm_band(img_np[ch] * PRITHVI_STD[ch] + PRITHVI_MEAN[ch])
    rgb = np.dstack([dn(2), dn(1), dn(0)])

    err = np.zeros((*mask_np.shape, 3))
    err[(pred_b==1)&(mask_np==1)] = [0.0, 0.8, 0.0]
    err[(pred_b==1)&(mask_np==0)] = [1.0, 0.5, 0.0]
    err[(pred_b==0)&(mask_np==1)] = [0.9, 0.1, 0.1]

    axes[row,0].imshow(rgb);                                    axes[row,0].axis('off')
    axes[row,1].imshow(mask_np, cmap='Reds', vmin=0, vmax=1);  axes[row,1].axis('off')
    axes[row,2].imshow(prob_np, cmap='hot',  vmin=0, vmax=1);  axes[row,2].axis('off')
    axes[row,3].imshow(err);                                    axes[row,3].axis('off')

for ax, t in zip(axes[0], ['RGB', 'Ground truth (dNBR)', 'Prediction (prob)', 'TP/FP/FN']):
    ax.set_title(t, fontsize=10)

tp_p = mpatches.Patch(color=[0,.8,0], label='TP')
fp_p = mpatches.Patch(color=[1,.5,0], label='FP')
fn_p = mpatches.Patch(color=[.9,.1,.1], label='FN')
fig.legend(handles=[tp_p,fp_p,fn_p], loc='lower center', ncol=3, bbox_to_anchor=(0.5,-0.01))
plt.suptitle(f'Córdoba — Fine-tuned | IoU base: {results_before["iou"]:.4f} → FT: {best_iou_ft:.4f}',
             fontsize=12, y=1.01)
plt.tight_layout()
out = RESULTS / 'cordoba_finetune_predictions.png'
plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
print(f'Saved: {out}')

In [ ]:
# ── Resumen portfolio ─────────────────────────────────────────────────────────
print('='*65)
print('  PORTFOLIO SUMMARY — Few-Shot Domain Adaptation')
print('='*65)
print(f'  Modelo base  : Prithvi-100M + FPN (trained on Corrientes 2022)')
print(f'  Fine-tune    : {N_FINETUNE} Córdoba 2020 patches ({EPOCHS_FT} épocas)')
print(f'  Strategy     : encoder frozen, decoder only updated')
print('-'*65)
print(f'  {"Métrica":<12} {"Corrientes val":>14} {"Córdoba base":>14} {"Córdoba FT":>12}')
print('-'*65)
print(f'  {"IoU":<12} {0.5385:>14.4f} {results_before["iou"]:>14.4f} {best_iou_ft:>12.4f}')
print(f'  {"Recall":<12} {0.7068:>14.4f} {results_before["recall"]:>14.4f} {results_after["recall"]:>12.4f}')
print(f'  {"Precision":<12} {0.6934:>14.4f} {results_before["precision"]:>14.4f} {results_after["precision"]:>12.4f}')
print('='*65)